# Step 01. Spline 경로 생성

> **출처**: PythonRobotics (https://github.com/AtsushiSakai/PythonRobotics)  
> Original author: Atsushi Sakai (@Atsushi_twi) / License: MIT  
>  
> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

Day 1에서 Dijkstra / A\* / Hybrid A\*로 **Global Path**를 생성했습니다.  
Global Path는 격자 꺾임이 있어 차량이 바로 따라가기 어렵습니다.  
Day 2에서는 이 경로를 **차량이 실제로 주행 가능한 Trajectory**로 변환합니다.  
첫 단계는 경로를 **Cubic Spline**으로 부드럽게 보간하는 것입니다.

## 학습 목표

1. 격자 경로의 문제점과 C2 연속성의 필요성 이해
2. Cubic Spline 보간 원리 이해 (1D → 2D)
3. 호 길이(s), yaw, 곡률(κ) 계산 방법 이해
4. A\* 출력 → Spline 보간 → 부드러운 경로 변환 확인

In [ ]:
%matplotlib inline
import sys, os, math
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from utils.cubic_spline import CubicSpline1D, CubicSpline2D, calc_spline_course

print('라이브러리 로드 완료')
print(f'  numpy: {np.__version__}')


## 1. 왜 Spline이 필요한가?

### 이론 배경

A\* 가 출력하는 격자 경로는 방향이 불연속합니다.  
실제 차량이 따라가려면 **C2 연속성** (위치, 속도, 가속도 모두 연속)이 필요합니다.

| 연속성 | 보장 내용 | 위반 시 문제 |
|--------|-----------|-------------|
| C0 | 위치 연속 | 경로가 끊김 |
| C1 | 속도(방향) 연속 | 급격한 방향 변화 (조향 급변) |
| C2 | 가속도(곡률) 연속 | 급격한 횡방향 가속도 (승차감 저하) |

**Cubic Spline**은 구간마다 3차 다항식 $f(x) = ax^3+bx^2+cx+d$를 사용하며,  
경계 조건으로 C2 연속성을 자동으로 보장합니다.

In [ ]:
%matplotlib inline
import sys, os, math
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from utils.cubic_spline import CubicSpline1D, CubicSpline2D, calc_spline_course

print('라이브러리 로드 완료')


## 2. CubicSpline1D — 1차원 스플라인

### 이론 배경

구간 $[x_i, x_{i+1}]$ 에서 3차 다항식 $S_i(x) = a_i x^3 + b_i x^2 + c_i x + d_i$

**계수 결정 조건:**
- **내부 경계**: $S_i(x_{i+1}) = S_{i+1}(x_{i+1})$, $S'_i = S'_{i+1}$, $S''_i = S''_{i+1}$
- **자연 경계**: 양 끝점에서 $S'' = 0$ (경계에서 곡률 없음)

이 조건을 트리다이어그널 선형 시스템으로 풀어 계수를 결정합니다.  
결과로 `y'(위치)`, `y''(기울기)`, `y'''(곡률 비례)` 가 모두 연속이 됩니다.

In [ ]:
wp_x = [0.0, 10.0, 20.0, 30.0, 40.0]
wp_y = [0.0,  5.0, -2.0,  8.0,  3.0]

sp = CubicSpline1D(wp_x, wp_y)

t_dense  = np.arange(0, 40.1, 0.1)
y_dense  = [sp.calc_position(t)         for t in t_dense]
y1_dense = [sp.calc_first_derivative(t) for t in t_dense]
y2_dense = [sp.calc_second_derivative(t)for t in t_dense]

print('CubicSpline1D 보간 확인')
print(f'  {"x":>6}  {"원본 y":>8}  {"스플라인 y":>12}  {"오차":>8}')
print('  ' + '-' * 42)
for xi, yi in zip(wp_x, wp_y):
    sp_val = sp.calc_position(xi)
    print(f'  {xi:>6.1f}  {yi:>8.3f}  {sp_val:>12.6f}  {abs(yi-sp_val):>8.2e}')

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
axes[0].plot(wp_x, wp_y, 'o', ms=8, color='orange', label='Control Points')
axes[0].plot(t_dense, y_dense, '-b', lw=2, label='Spline')
axes[0].set_ylabel('y'); axes[0].legend(); axes[0].set_title('CubicSpline1D')
axes[1].plot(t_dense, y1_dense, '-g', lw=1.5)
axes[1].set_ylabel("y' (1st derivative)"); axes[1].axhline(0, color='k', lw=0.5)
axes[2].plot(t_dense, y2_dense, '-r', lw=1.5)
axes[2].set_ylabel("y'' (2nd derivative)"); axes[2].axhline(0, color='k', lw=0.5)
axes[2].set_xlabel('x')
for ax in axes: ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('1차 미분(기울기)과 2차 미분(곡률 비례)이 연속 -> C2 연속성 확인')


## 3. CubicSpline2D — 2차원 경로 보간

### 이론 배경

2D 경로 $(x(s), y(s))$는 **호 길이 s**를 매개변수로 사용합니다.

$$s_{i+1} = s_i + \sqrt{(\Delta x)^2 + (\Delta y)^2}$$

각 좌표를 s에 대한 1D Spline으로 독립적으로 보간한 뒤 조합합니다.  

**곡률** $\kappa$는 다음으로 계산합니다:
$$\kappa = \frac{x'y'' - y'x''}{(x'^2 + y'^2)^{1.5}}$$

곡률이 큰 구간은 Step 03에서 속도 제한의 근거가 됩니다.

In [ ]:
wx = [0.0, 10.0, 20.5, 35.0, 50.0]
wy = [0.0, -6.0,  5.0,  6.5,  0.0]

rx, ry, ryaw, rk, rs = calc_spline_course(wx, wy, ds=0.5)

max_curv = max(abs(k) for k in rk)
print('CubicSpline2D 결과')
print(f'  제어점 개수    : {len(wx)}개')
print(f'  경로 점 개수   : {len(rx)}개 (ds=0.5m)')
print(f'  총 경로 길이   : {rs[-1]:.2f} m')
print(f'  최대 곡률      : {max_curv:.4f} rad/m')
print(f'  최소 곡률반경  : {1/max_curv:.2f} m')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(wx, wy, 'o-', ms=8, color='orange', label='Waypoints')
axes[0].plot(rx, ry, '-b', lw=2, label='Spline Path')
axes[0].set_aspect('equal'); axes[0].legend()
axes[0].grid(True, alpha=0.3); axes[0].set_title('2D Spline Path')
axes[1].plot(rs, rk, '-r', lw=2)
axes[1].set_xlabel('Arc length s [m]'); axes[1].set_ylabel('Curvature kappa [rad/m]')
axes[1].axhline(0, color='k', lw=0.5)
axes[1].grid(True, alpha=0.3); axes[1].set_title('Curvature Profile')
plt.tight_layout()
plt.show()
print('곡률이 급변하는 구간 = 속도 제한 지점 (Step 03에서 활용)')


## 4. Raw 격자 경로 vs Spline 비교

### 이론 배경

A\* 출력 → Spline 보간 파이프라인:

```
A* 출력 (격자 경유점)  -->  Spline 보간  -->  부드러운 경로
   꺾임 있음                  C2 연속                주행 가능
```

Spline은 웨이포인트를 **통과(interpolation)**하면서 부드럽게 연결합니다.  
(회귀(regression)와 다르게 모든 웨이포인트를 정확히 지남)

In [ ]:
raw_x = [0,  0,  0,  5, 10, 15, 20, 25, 30, 35, 40, 50]
raw_y = [0, -2, -5, -6, -6, -4,  0,  3,  6,  6,  4,  0]

rx2, ry2, _, _, _ = calc_spline_course(raw_x, raw_y, ds=0.5)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(raw_x, raw_y, 'o-r', lw=1.5, ms=5, label='Raw Path (A* output)')
axes[0].set_aspect('equal'); axes[0].grid(True, alpha=0.3)
axes[0].legend(); axes[0].set_title('Raw Path — Sharp Corners')
axes[1].plot(raw_x, raw_y, 'o', ms=5, color='orange', alpha=0.6, label='Waypoints')
axes[1].plot(rx2, ry2, '-b', lw=2, label='Spline Interpolated')
axes[1].set_aspect('equal'); axes[1].grid(True, alpha=0.3)
axes[1].legend(); axes[1].set_title('Spline Path — Smooth Curve')
plt.tight_layout()
plt.show()
print('Spline은 A* 격자 경로 웨이포인트를 통과하면서 곡률이 연속인 경로 생성')


## 5. 차량 경로 추종 시각화

### 이론 배경

각 경로 점에서 yaw, 곡률(κ), 호 길이(s)가 계산됩니다.  
이 정보들이 Day 2의 나머지 단계(속도 계획, Frenet 변환)의 입력이 됩니다.

- `yaw`: 차량 방향 → 조향 제어에 사용
- `kappa (κ)`: 곡률 → Step 03 속도 제한에 사용
- `s`: 호 길이 → Step 02 Frenet 좌표의 종방향 거리

In [ ]:
# 차량(점)이 spline 경로를 따라 이동하는 애니메이션
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(wx, wy, 'o', ms=8, color='orange', alpha=0.7, label='Waypoints', zorder=3)
ax.plot(rx, ry, '-', color='lightblue', lw=2, alpha=0.6, label='Spline Path')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend(loc='upper left')
ax.set_title('Vehicle Moving Along Spline Path')

dot, = ax.plot([], [], 'o', color='royalblue', ms=10, zorder=5)
trail, = ax.plot([], [], '-', color='royalblue', lw=1.5, alpha=0.5)
heading = ax.annotate('', xy=(0,0), xytext=(0,0),
                      arrowprops=dict(arrowstyle='->', color='red', lw=2))
info_text = ax.text(0.02, 0.95, '', transform=ax.transAxes,
                    fontsize=9, verticalalignment='top', color='navy')
SKIP = 3  # 프레임마다 SKIP 포인트 이동

def update_vehicle(frame):
    k = min(frame * SKIP, len(rx)-1)
    dot.set_data([rx[k]], [ry[k]])
    trail.set_data(rx[:k+1], ry[:k+1])
    heading.set_position((rx[k], ry[k]))
    dx = math.cos(ryaw[k]) * 2
    dy = math.sin(ryaw[k]) * 2
    heading.xy = (rx[k]+dx, ry[k]+dy)
    info_text.set_text(f's={rs[k]:.1f}m  yaw={math.degrees(ryaw[k]):.1f}deg  kappa={rk[k]:.4f}')
    return dot, trail, info_text

n_frames = math.ceil(len(rx) / SKIP) + 5
ani_sp = FuncAnimation(fig, update_vehicle, frames=n_frames,
                       interval=50, repeat=False, blit=False)
HTML(ani_sp.to_jshtml())
